In [6]:
import pandas as pd
import json
import random

In [61]:
result_dict = """<think>\nThe user wants me to create a high-performance CUDA extension for their PyTorch model. Let me analyze the reference code first.\n\nThe original Model class does:\n1. BatchNorm3d(10) - 3D batch normalization\n2. digamma(x) - Digamma function (ψ(x))\n3. max(x) - Maximum value\n4. Adds a learnable parameter\n\nThe task is to create a CUDA extension that's faster than the baseline.\n\nLet me plan the CUDA kernels:\n\n1. BatchNorm3d - cuDNN is mandatory for this. I'll use cuDNN's batch normalization API.\n\n2. Digamma function - This is tricky. The digamma function doesn't have a simple CUDA implementation. I'll need to create a CUDA kernel that computes ψ(x) efficiently, possibly using approximation formulas.\n\n3. Max pooling/reduction - This can be done with CUDA thrust or a custom reduction kernel.\n\n4. Addition with parameter - Simple element-wise addition.\n\nLet me structure this:\n\nCUDA_KERNELS:\n- batch_norm_kernel.cu - Use cuDNN for batch normalization\n- digamma_kernel.cu - Custom kernel for digamma function\n- reduction_kernel.cu - Custom kernel for max reduction\n\nAPPLY_BINDINGS:\n- batch_norm.cpp - Wrapper for cuDNN batch norm\n- digamma.cpp - Wrapper for digamma kernel\n- reduction.cpp - Wrapper for max reduction\n- model.cpp - Complete model pipeline\n\nMODEL_NEW:\n- ModelNew class that uses the custom ops\n\nLet me write the code:\n\n### CUDA_KERNELS\n\nbatch_norm.cu:\n```cpp\n#include <cuda_runtime.h>\n#include <cudnn.h>\n#include <cassert>\n\nextern \"C\" void batch_norm_forward_launcher(\n    float* output,\n    const float* input,\n    const float* weight,\n    const float* bias,\n    const float* running_mean,\n    const float* running_var,\n    float eps,\n    int batch_size,\n    int channels,\n    int depth,\n    int height,\n    int width,\n    cudaStream_t stream\n) {\n    // cuDNN batch normalization setup\n    cudnnHandle_t handle;\n    cudnnCreate(&handle);\n    \n    // Tensor descriptors\n    cudnnTensorDescriptor_t input_desc, output_desc;\n    cudnnCreateTensorDescriptor(&input_desc);\n    cudnnCreateTensorDescriptor(&output_desc);\n    \n    int input_dims[] = {batch_size, channels, depth, height, width};\n    int strides[5] = {channels * depth * height * width,\n                      depth * height * width,\n                      height * width,\n                      width,\n                      1};\n    \n    cudnnSetTensorNCHWv2(input_desc, CUDNN_TENSOR_NCHW, \n                         channels, depth, height, width, \n                         strides[1], strides[2], strides[3], strides[4]);\n    cudnnSetTensorNCHWv2(output_desc, CUDNN_TENSOR_NCHW, \n                         channels, depth, height, width, \n                         strides[1], strides[2], strides[3], strides[4]);\n    \n    // Batch normalization descriptor\n    cudnnBatchNormalizationDescriptor_t bn_desc;\n    cudnnCreateBatchNormalizationDescriptor(&bn_desc);\n    cudnnSetBatchNormalizationDescriptor(bn_desc, CUDNN_BATCHNORM_SPATIAL,\n                                        channels, eps, nullptr, nullptr);\n    \n    // Execution\n    cudnnBatchNormalizationForwardInference(handle, \n                                          CUDNN_BATCHNORM_SPATIAL,\n                                          input_desc, input,\n                                          output_desc, output,\n                                          bn_desc, weight, bias,\n                                          running_mean, running_var,\n                                          eps);\n    \n    // Clean up\n    cudnnDestroyTensorDescriptor(input_desc);\n    cudnnDestroyTensorDescriptor(output_desc);\n    cudnnDestroyBatchNormalizationDescriptor(bn_desc);\n    cudnnDestroy(handle);\n}\n```\n\ndigamma.cu:\n```cpp\n#include <cuda_runtime.h>\n#include <math_constants.h>\n\n// Digamma approximation for x > 0\n__device__ float digamma_approx(float x) {\n    // Use the approximation from Abramowitz and Stegun\n    // ψ(x) = ln(x) - 1/(2x) - 1/(12x^2) + 1/(120x^4) - ...\n    \n    if (x < 0.5) {\n        // Reflection formula: ψ(1-x) = ψ(x) + π cot(πx)\n        // For x < 0.5, use recurrence ψ(x) = ψ(x+1) - 1/x\n        return digamma_approx(x + 1) - 1.0f / x;\n    }\n    \n    if (x == 1.0f) return -EULER_MASCHERONI;\n    \n    float result = logf(x) - 1.0f / (2.0f * x);\n    result -= 1.0f / (12.0f * x * x);\n    result += 1.0f / (120.0f * x * x * x * x);\n    \n    return result;\n}\n\n__global__ void digamma_kernel(float* output, const float* input, int size) {\n    int tid = blockIdx.x * blockDim.x + threadIdx.x;\n    int stride = blockDim.x * gridDim.x;\n    \n    for (int i = tid; i < size; i += stride) {\n        output[i] = digamma_approx(input[i]);\n    }\n}\n\nextern \"C\" void digamma_launcher(\n    float* output,\n    const float* input,\n    int size,\n    cudaStream_t stream\n) {\n    int threads = 256;\n    int blocks = (size + threads - 1) / threads;\n    \n    digamma_kernel<<<blocks, threads, 0, stream>>>(output, input, size);\n}\n```\n\nreduction.cu:\n```cpp\n#include <cuda_runtime.h>\n#include <thrust/device_vector.h>\n#include <thrust/functional.h>\n#include <thrust/extrema.h>\n\n__global__ void max_reduction(float* output, const float* input, int size) {\n    __shared__ float sdata[256];\n    int tid = threadIdx.x;\n    int bid = blockIdx.x;\n    int idx = tid + bid * blockDim.x;\n    \n    // Initial value\n    float my_max = -FLT_MAX;\n    \n    // Find max in warp\n    for (int i = tid; i < size; i += blockDim.x) {\n        my_max = fmaxf(my_max, input[i]);\n    }\n    \n    // Warp reduce\n    for (int mask = 16; mask > 0; mask >>= 1) {\n        float other = __shfl_down(my_max, mask);\n        my_max = fmaxf(my_max, other);\n    }\n    \n    if (tid == 0) {\n        output[bid] = my_max;\n    }\n}\n\nextern \"C\" void max_reduction_launcher(\n    float* output,\n    const float* input,\n    int size,\n    int blockDim,\n    cudaStream_t stream\n) {\n    int blocks = (size + blockDim - 1) / blockDim;\n    \n    max_reduction<<<blocks, blockDim, 0, stream>>>(\n        output, input, size\n    );\n}\n```\n\n### APPLY_BINDINGS\n\nbatch_norm.cpp:\n```cpp\n#include <torch/types.h>\n#include <torch/csrc/utils/pybind.h>\n#include <c10/cuda/CUDAStream.h>\n#include \"../binding_registry.h\"\n\nextern \"C\" void batch_norm_forward_launcher(\n    float* output,\n    const float* input,\n    const float* weight,\n    const float* bias,\n    const float* running_mean,\n    const float* running_var,\n    float eps,\n    int batch_size,\n    int channels,\n    int depth,\n    int height,\n    int width,\n    cudaStream_t stream\n);\n\ntorch::Tensor batch_norm_forward(torch::Tensor input, \n                                torch::Tensor weight,\n                                torch::Tensor bias,\n                                torch::Tensor running_mean,\n                                torch::Tensor running_var,\n                                float eps) {\n    TORCH_CHECK(input.is_cuda(), \"Input must be CUDA tensor\");\n    TORCH_CHECK(input.is_contiguous(), \"Input must be contiguous\");\n    TORCH_CHECK(input.dtype() == torch::kFloat32, \"Input must be float32\");\n    \n    auto output = torch::empty_like(input);\n    cudaStream_t stream = c10::cuda::getCurrentCUDAStream().stream();\n    \n    batch_norm_forward_launcher(\n        output.data_ptr<float>(),\n        input.data_ptr<float>(),\n        weight.data_ptr<float>(),\n        bias.data_ptr<float>(),\n        running_mean.data_ptr<float>(),\n        running_var.data_ptr<float>(),\n        eps,\n        input.size(0),\n        input.size(1),\n        input.size(2),\n        input.size(3),\n        input.size(4),\n        stream\n    );\n    \n    return output;\n}\n\nvoid register_batch_norm(pybind11::module& m) {\n    m.def(\"batch_norm_forward\", &batch_norm_forward,\n          \"CUDA batch normalization forward\",\n          py::arg(\"input\"), py::arg(\"weight\"), py::arg(\"bias\"),\n          py::arg(\"running_mean\"), py::arg(\"running_var\"),\n          py::arg(\"eps\"));\n}\n\nREGISTER_BINDING(batch_norm, register_batch_norm);\n```\n\ndigamma.cpp:\n```cpp\n#include <torch/types.h>\n#include <torch/csrc/utils/pybind.h>\n#include <c10/cuda/CUDAStream.h>\n#include \"../binding_registry.h\"\n\nextern \"C\" void digamma_launcher(\n    float* output,\n    const float* input,\n    int size,\n    cudaStream_t stream\n);\n\ntorch::Tensor digamma_forward(torch::Tensor input) {\n    TORCH_CHECK(input.is_cuda(), \"Input must be CUDA tensor\");\n    TORCH_CHECK(input.is_contiguous(), \"Input must be contiguous\");\n    TORCH_CHECK(input.dtype() == torch::kFloat32, \"Input must be float32\");\n    \n    auto output = torch::empty_like(input);\n    cudaStream_t stream = c10::cuda::getCurrentCUDAStream().stream();\n    \n    digamma_launcher(\n        output.data_ptr<float>(),\n        input.data_ptr<float>(),\n        input.numel(),\n        stream\n    );\n    \n    return output;\n}\n\nvoid register_digamma(pybind11::module& m) {\n    m.def(\"digamma_forward\", &digamma_forward,\n          \"CUDA digamma function\",\n          py::arg(\"input\"));\n}\n\nREGISTER_BINDING(digamma, register_digamma);\n```\n\nreduction.cpp:\n```cpp\n#include <torch/types.h>\n#include <torch/csrc/utils/pybind.h>\n#include <c10/cuda/CUDAStream.h>\n#include \"../binding_registry.h\"\n\nextern \"C\" void max_reduction_launcher(\n    float* output,\n    const float* input,\n    int size,\n    int blockDim,\n    cudaStream_t stream\n);\n\ntorch::Tensor max_forward(torch::Tensor input) {\n    TORCH_CHECK(input.is_cuda(), \"Input must be CUDA tensor\");\n    TORCH_CHECK(input.is_contiguous(), \"Input must be contiguous\");\n    TORCH_CHECK(input.dtype() == torch::kFloat32, \"Input must be float32\");\n    \n    int size = input.numel();\n    int blockDim = 256;\n    int blocks = (size + blockDim - 1) / blockDim;\n    \n    auto output = torch::empty({blocks}, torch::dtype(torch::kFloat32).device(input.device()));\n    \n    cudaStream_t stream = c10::cuda::getCurrentCUDAStream().stream();\n    \n    max_reduction_launcher(\n        output.data_ptr<float>(),\n        input.data_ptr<float>(),\n        size,\n        blockDim,\n        stream\n    );\n    \n    return output;\n}\n\nvoid register_max(pybind11::module& m) {\n    m.def(\"max_forward\", &max_forward,\n          \"CUDA max reduction\",\n          py::arg(\"input\"));\n}\n\nREGISTER_BINDING(max_op, register_max);\n```\n\nmodel.cpp:\n```cpp\n#include <torch/types.h>\n#include <torch/csrc/utils/pybind.h>\n#include <c10/cuda/CUDAStream.h>\n#include \"../binding_registry.h\"\n\n#include <vector>\n\ntorch::Tensor model_forward(torch::Tensor input,\n                          torch::Tensor weight,\n                          torch::Tensor bias,\n                          torch::Tensor running_mean,\n                          torch::Tensor running_var,\n                          float eps) {\n    // Batch normalization\n    auto bn_output = batch_norm_forward(input, weight, bias, running_mean, running_var, eps);\n    \n    // Digamma\n    auto digamma_output = digamma_forward(bn_output);\n    \n    // Max reduction\n    auto max_output = max_forward(digamma_output);\n    \n    // Return max_output (scalar)\n    return max_output;\n}\n\nvoid register_model(pybind11::module& m) {\n    m.def(\"model_forward\", &model_forward,\n          \"Complete model forward\",\n          py::arg(\"input\"), py::arg(\"weight\"), py::arg(\"bias\"),\n          py::arg(\"running_mean\"), py::arg(\"running_var\"),\n          py::arg(\"eps\"));\n}\n\nREGISTER_BINDING(model, register_model);\n```\n\n### MODEL_NEW\n\nmodel_new.py:\n```python\nimport torch\nfrom torch.nn import Parameter\n\n\nclass ModelNew(torch.nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.batch_norm = torch.nn.BatchNorm3d(10)\n        self.parameter = Parameter(torch.randn(10))\n        \n        # Pre-compile custom ops\n        import cuda_extension\n        \n    def forward(self, x):\n        # Batch normalization (custom cuDNN version)\n        x = self.batch_norm(x)\n        \n        # Digamma - custom CUDA kernel\n        x = torch.ops.cuda_extension.digamma_forward(x)\n        \n        # Max reduction - custom CUDA kernel\n        x = torch.ops.cuda_extension.max_forward(x)\n        \n        # Add learnable parameter\n        x = x + self.parameter\n        \n        return x\n\n\nbatch_size = 512\nchannels = 10\ndepth = 15\nheight = 15\nwidth = 15\n\n\ndef get_inputs():\n    return [torch.randn(batch_size, channels, depth, height, width).cuda()]\n\n\ndef get_init_inputs():\n    return []\n```\n\nI'll verify the code structure and make sure it's complete and compilable. The key points are:\n1. Using cuDNN for batch normalization (much faster than PyTorch's default)\n2. Custom CUDA kernels for digamma (approximation) and max reduction\n3. Proper tensor memory management and device synchronization\n4. Python bindings using PyBind11\n\nThe code should provide significant performance improvements over the original PyTorch implementation.\n</think>\n\n # PyTorch Model Acceleration with CUDA Extension\n\n### CUDA_KERNELS\n```cpp\n#include <cuda_runtime.h>\n#include <cudnn.h>\n#include <cmath>\n#include <cassert>\n\n// ============================ //\n// BatchNorm3d CUDA Kernel    //\n// ============================ //\n\n__global__ void batchNorm3dKernel(\n    const float* input, float* output,\n    const float* weight, const float* bias,\n    const float* running_mean, const float* running_var,\n    float eps, int batch_size, int channels,\n    int depth, int height, int width\n) {\n    int n = blockIdx.x;\n    int c = blockIdx.y;\n    int d = threadIdx.x;\n    int h = threadIdx.y;\n    int w = threadIdx.z;\n    \n    int idx_in = n * channels * depth * height * width +\n                 c * depth * height * width +\n                 d * height * width +\n                 h * width + w;\n                 \n    float x = input[idx_in];\n    \n    float mean = running_mean[c];\n    float var = running_var[c];\n    \n    float x_centered = x - mean;\n    float x_normalized = x_centered / sqrt(var + eps);\n    \n    float gamma = weight[c];\n    float beta = bias[c];\n    \n    output[idx_in] = gamma * x_normalized + beta;\n}\n\nextern \"C\" void batchNorm3dLauncher(\n    const float* input, float* output,\n    const float* weight, const float* bias,\n    const float* running_mean, const float* running_var,\n    float eps, int batch_size, int channels,\n    int depth, int height, int width,\n    cudaStream_t stream\n) {\n    dim3 blockDim(8, 4, 2);\n    dim3 gridDim(batch_size, channels, 1);\n    \n    batchNorm3dKernel<<<gridDim, blockDim, 0, stream>>>(\n        input, output, weight, bias,\n        running_mean, running_var, eps,\n        batch_size, channels, depth, height, width\n    );\n    \n    cudaError_t err = cudaGetLastError();\n    if (err != cudaSuccess) {\n        fprintf(stderr, \"CUDA error in batchNorm3dLauncher: %s\\n\", cudaGetErrorString(err));\n        exit(1);\n    }\n}\n\n// ============================ //\n// Digamma Function Kernel    //\n// ============================ //\n\n__device__ float digammaApprox(float x) {\n    // Digamma function approximation using asymptotic expansion\n    // ψ(x) = ln(x) - 1/(2x) - 1/(12x^2) + 1/(120x^4) - ...\n    \n    if (x < 0.5f) {\n        // Use reflection formula: ψ(1-x) = ψ(x) + π cot(πx)\n        // For x < 0.5, use recurrence ψ(x) = ψ(x+1) - 1/x\n        return digammaApprox(x + 1.0f) - 1.0f / x;\n    }\n    \n    if (x == 1.0f) return -0.5772136857401516f; // Euler-Mascheroni constant\n    \n    float result = logf(x);\n    result -= 1.0f / (2.0f * x);\n    result -= 1.0f / (12.0f * x * x);\n    result += 1.0f / (120.0f * x * x * x * x);\n    \n    return result;\n}\n\n__global__ void digammaKernel(float* output, const float* input, int size) {\n    int tid = threadIdx.x + blockIdx.x * blockDim.x;\n    if (tid < size) {\n        output[tid] = digammaApprox(input[tid]);\n    }\n}\n\nextern \"C\" void digammaLauncher(\n    float* output, const float* input, int size,\n    cudaStream_t stream\n) {\n    int threads = 256;\n    int blocks = (size + threads - 1) / threads;\n    \n    digammaKernel<<<blocks, threads, 0, stream>>>(\n        output, input, size\n    );\n    \n    cudaError_t err = cudaGetLastError();\n    if (err != cudaSuccess) {\n        fprintf(stderr, \"CUDA error in digammaLauncher: %s\\n\", cudaGetErrorString(err));\n        exit(1);\n    }\n}\n\n// ============================ //\n// Max Reduction Kernel       //\n// ============================ //\n\n__global__ void maxReductionKernel(\n    float* output, const float* input, int size\n) {\n    __shared__ float sdata[256];\n    int tid """

In [62]:
print(result_dict)

<think>
The user wants me to create a high-performance CUDA extension for their PyTorch model. Let me analyze the reference code first.

The original Model class does:
1. BatchNorm3d(10) - 3D batch normalization
2. digamma(x) - Digamma function (ψ(x))
3. max(x) - Maximum value
4. Adds a learnable parameter

The task is to create a CUDA extension that's faster than the baseline.

Let me plan the CUDA kernels:

1. BatchNorm3d - cuDNN is mandatory for this. I'll use cuDNN's batch normalization API.

2. Digamma function - This is tricky. The digamma function doesn't have a simple CUDA implementation. I'll need to create a CUDA kernel that computes ψ(x) efficiently, possibly using approximation formulas.

3. Max pooling/reduction - This can be done with CUDA thrust or a custom reduction kernel.

4. Addition with parameter - Simple element-wise addition.

Let me structure this:

CUDA_KERNELS:
- batch_norm_kernel.cu - Use cuDNN for batch normalization
- digamma_kernel.cu - Custom kernel for d

In [11]:
coldstart_data = pd.read_parquet("/nfs/FM/gongoubo/cuda_kernel/drkernel-coldstart-8k/drkernel-coldstart-8k.parquet")


In [12]:
print(coldstart_data.loc[0]['messages'][0]['content'])


You write custom Triton kernels to replace the pytorch operators in the given architecture to get speedups. 

    You have complete freedom to choose the set of operators you want to replace. You may make the decision to replace some operators with custom Triton kernels and leave others unchanged. You may replace multiple operators with custom implementations, consider operator fusion opportunities (combining multiple operators into a single kernel, for example, combining matmul+relu), or algorithmic changes (such as online softmax). You are only limited by your imagination.


        Here's an example to show you the syntax of inline embedding custom Triton kernels in torch: The example given architecture is:

        ```

        import torch
import torch.nn as nn
import torch.nn.functional as F


class Model(nn.Module):
    def __init__(self) -> None:
        super().__init__()

    def forward(self, a, b):
        return a + b


def get_inputs():
    # randomly generate input tenso

In [7]:
drkernel_rl_data = pd.read_parquet("/nfs/FM/lihongbin/datasets/drkernel-rl-data/cuda_llm_rl_thinking_1025.parquet")
drkernel_vaild_data = pd.read_parquet("/nfs/FM/lihongbin/datasets/drkernel-validation-data/validation_data_thinking.parquet")
cuda_agent_data = pd.read_parquet("/nfs/FM/lihongbin/datasets/CUDA-Agent-Ops-6K/data.parquet")

In [8]:
print(drkernel_rl_data.columns)
print(drkernel_vaild_data.columns)
print(cuda_agent_data.columns)

Index(['data_source', 'prompt', 'ability', 'reward_model', 'extra_info'], dtype='object')
Index(['data_source', 'prompt', 'reward_model', 'ability', 'extra_info'], dtype='object')
Index(['ops', 'data_source', 'code'], dtype='object')


In [9]:
cuda_agent_data.head()

,ops,data_source,code
0,"[""nn.BatchNorm3d"", ""torch.diag"", ""torch.max"", ...",torch#4,"import torch\nfrom torch import digamma, max\n..."
1,"[""nn.LeakyReLU"", ""torch.pow"", ""torch.abs"", ""to...",torch#5,import torch\nimport torch.nn as nn\nimport to...
2,"[""torch.ceil"", ""torch.transpose"", ""F.group_norm""]",torch#3,import torch\nimport torch.nn as nn\nimport to...
3,"[""torch.multiply"", ""torch.stack"", ""nn.Multihea...",torch#3,import torch\nimport torch.nn as nn\n\n\nclass...
4,"[""torch.repeat_interleave"", ""nn.MultiheadAtten...",torch#3,import torch\nimport torch.nn as nn\n\n\nclass...


In [64]:
drkernel_rl_data.iloc[0]['reward_model']['ground_truth'] in drkernel_rl_data.iloc[0]['prompt'][0]['content']

True

In [67]:
print(drkernel_vaild_data.iloc[0]['reward_model']['ground_truth'])


import torch
import torch.nn as nn

class Model(nn.Module):
    """
    A model that performs a transposed 3D convolution, clamps the output to a minimum value, 
    and then divides the result by a constant.
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding, min_value, divisor):
        super(Model, self).__init__()
        self.conv_transpose = nn.ConvTranspose3d(in_channels, out_channels, kernel_size, stride=stride, padding=padding)
        self.min_value = min_value
        self.divisor = divisor

    def forward(self, x):
        x = self.conv_transpose(x)
        x = torch.clamp(x, min=self.min_value)
        x = x / self.divisor
        return x

batch_size = 16
in_channels = 64
out_channels = 128
depth, height, width = 24, 48, 48
kernel_size = 3
stride = 2
padding = 1
min_value = -1.0
divisor = 2.0

def get_inputs():
    return [torch.rand(batch_size, in_channels, depth, height, width)]

def get_init_inputs():
    return [in_channels, out_ch

In [68]:
print(drkernel_vaild_data.iloc[0]['prompt'][0]['content'])


You write custom Triton kernels to replace the pytorch operators in the given architecture to get speedups. 

    You have complete freedom to choose the set of operators you want to replace. You may make the decision to replace some operators with custom Triton kernels and leave others unchanged. You may replace multiple operators with custom implementations, consider operator fusion opportunities (combining multiple operators into a single kernel, for example, combining matmul+relu), or algorithmic changes (such as online softmax). You are only limited by your imagination.


        Here's an example to show you the syntax of inline embedding custom Triton kernels in torch: The example given architecture is:

        ```

        import torch
import torch.nn as nn
import torch.nn.functional as F


class Model(nn.Module):
    def __init__(self) -> None:
        super().__init__()

    def forward(self, a, b):
        return a + b


def get_inputs():
    # randomly generate input tenso

In [15]:
print(rl_data.iloc[0]["reward_model"]["ground_truth"])


import torch
import torch.nn as nn
import torch.nn.functional as F


class Model(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding, dilation, bias):
        super().__init__()
        self.conv2d = nn.Conv2d(
            in_channels, out_channels, kernel_size, stride=stride, padding=padding, dilation=dilation, bias=bias
        )

    def forward(self, x):
        x = self.conv2d(x)
        x = torch.log2(torch.abs(x) + 1e-8)  # log2 with small epsilon to prevent log of zero
        return x


batch_size = 64
in_channels = 16
out_channels = 32
kernel_size = 3
stride = 1
padding = 1
dilation = 1
bias = False


def get_inputs():
    height, width = 256, 256
    return [torch.randn(batch_size, in_channels, height, width)]


def get_init_inputs():
    return [in_channels, out_channels, kernel_size, stride, padding, dilation, bias]



In [21]:
print("You are a PyTorch and CUDA expert. Accelerate the given PyTorch Model by creating a high-performance CUDA C++ extension, targeting the best possible performance faster than baseline.\n\n========================\n⚠️ STRICTLY FORBIDDEN\n========================\nNO torch operators in C++: NEVER use torch::* or torch::nn::functional::* in binding.cpp or .cu files\nNO torch operations in model_new.py: Only tensor creation and your custom ops allowed\nNO third-party libraries: Except cuBLAS (GEMM only) and cuDNN (Conv only)\n\n========================\n✅ ALLOWED ONLY\n========================\nC++: Raw CUDA kernels (for custom ops), cuBLAS (for GEMM), cuDNN (MANDATORY for Conv/ConvTranspose)\nPython: torch.tensor creation, custom extension ops, tensor properties (.shape, .device)\nMemory: torch::empty_like for allocation only\nFocus: Implement kernels in kernels/ directory only\n\n========================\nWORKSPACE STRUCTURE\n========================\n.\n├── binding_registry.h    # Do NOT modify - registration system\n├── binding.cpp           # Do NOT modify - main module binding\n├── kernels/              # YOUR WORK: Implement all kernels here\n├── model.py              # DO NOT modify - Original PyTorch model\n└── model_new.py          # YOUR WORK: Your optimized model using custom ops.\n\nFile Types and Usage\n.cu files: CUDA kernels with __global__ functions (custom implementations)\n.cpp files: cuDNN/cuBLAS API calls (NO custom kernels)\n_binding.cpp files: PyTorch tensor handling and Python bindings\n\n========================\nUNIFIED WORKFLOW\n========================\n\nwrite CUDA kernels with __global__ functions (custom implementations)\n```c++\n#include <cuda_runtime.h>\n\n// Template kernel for performance tuning\ntemplate<int BLOCK_SIZE, int TILE_SIZE>\n__global__ void my_kernel_impl(float* output, const float* input, int size) {\n    // Shared memory for tiling\n    extern __shared__ float smem[];\n    \n    int tid = blockIdx.x * blockDim.x + threadIdx.x;\n    int stride = blockDim.x * gridDim.x;\n    \n    // Grid-stride loop for large data\n    for (int i = tid; i < size; i += stride) {\n        // Kernel logic with optimizations\n        output[i] = /* computation */;\n    }\n}\n\n// C-interface launcher (no PyTorch dependencies)\nextern \"C\" void my_kernel_launcher(\n    float* output,\n    const float* input,\n    int size,\n    int config,\n    cudaStream_t stream\n) {\n    // Dynamic configuration selection\n    int blocks = (size + 255) / 256;\n    int shared_mem_size = 0;\n    \n    switch(config) {\n        case 0: \n            shared_mem_size = 256 * sizeof(float);\n            my_kernel_impl<256, 16><<<blocks, 256, shared_mem_size, stream>>>(\n                output, input, size);\n            break;\n        case 1: \n            shared_mem_size = 128 * sizeof(float);\n            my_kernel_impl<128, 32><<<blocks, 128, shared_mem_size, stream>>>(\n                output, input, size);\n            break;\n        default:\n            my_kernel_impl<256, 16><<<blocks, 256, 0, stream>>>(\n                output, input, size);\n    }\n}\n```\n\nwrite apply_bings.cpp (PyTorch tensor handling and Python bindings)\n```c++\n// Use this two headers to replace torch/extension.h for faster compilation\n#include <torch/types.h>\n#include <torch/csrc/utils/pybind.h>\n\n#include <cuda_runtime.h>\n#include <c10/cuda/CUDAStream.h>\n#include \"../binding_registry.h\"\n\n// Declare launcher from .cu file\nextern \"C\" void my_kernel_launcher(\n    float* output,\n    const float* input,\n    int size,\n    int config,\n    cudaStream_t stream\n);\n\n// PyTorch wrapper with config parameter\ntorch::Tensor my_kernel_forward(torch::Tensor input, int config = 0) {\n    // Input validation\n    TORCH_CHECK(input.is_cuda(), \"Input must be a CUDA tensor\");\n    TORCH_CHECK(input.is_contiguous(), \"Input must be contiguous\");\n    TORCH_CHECK(input.dtype() == torch::kFloat32, \"Input must be float32\");\n    \n    auto output = torch::empty_like(input);\n    \n    // Get current CUDA stream (correct way)\n    cudaStream_t stream = c10::cuda::getCurrentCUDAStream().stream();\n    \n    // Call CUDA launcher with config\n    my_kernel_launcher(\n        output.data_ptr<float>(),\n        input.data_ptr<float>(),\n        input.numel(),\n        config,\n        stream\n    );\n    \n    return output;\n}\n\n// Registration function\nvoid register_my_kernel(pybind11::module& m) {\n    m.def(\"my_kernel_forward\", &my_kernel_forward, \n          \"My kernel forward\",\n          py::arg(\"input\"),\n          py::arg(\"config\") = 0);\n}\n\n`#include \"../binding_registry.h\"` is needed! \n\n// Auto-register\nREGISTER_BINDING(my_kernel, register_my_kernel);\n```\n\ncreate model_new.py\n```\nimport torch\nimport torch.nn as nn\nimport cuda_extension\n\nclass ModelNew(nn.Module):\n    def __init__(self, ...):  # MUST match Model signature exactly\n        super().__init__()\n        # Initialize parameters - preserve original structure for state_dict compatibility\n        self.weight = nn.Parameter(torch.randn(...))\n        self.bias = nn.Parameter(torch.zeros(...))\n        \n    def forward(self, x):\n        # Use custom ops only - NO torch operations\n        x = cuda_extension.my_kernel_forward(x, config=0)\n        x = cuda_extension.gemm_forward(x, self.weight, self.bias)\n        return x\n```\n\n\n========================\n📦 OUTPUT FORMAT (STRICT MARKDOWN)\n========================\n\nYou MUST follow this EXACT format:\n\n### CUDA_KERNELS\n```cpp\n<CUDA .cu code here>\n\n### APPLY_BINDINGS\n```cpp\n<apply_bindings.cpp code here>\n```\n\n### MODEL_NEW\n```python\n<model_new.py code here>\n```\n\nBefore output:\n- Ensure all 3 sections exist\n- Ensure each section has valid code\n- Ensure no extra text outside sections\n\n\n\nreference pytorch code:\n```python\nimport torch\nimport torch.nn as nn\n\n\nclass Model(nn.Module):\n    def __init__(self):\n        super().__init__()\n\n    def forward(self, x):\n        x = torch.relu(x)  # Apply ReLU activation\n        x = x / 2.0  # Apply division by 2\n        return x\n\n\ndef get_inputs():\n    return [torch.randn(64, 128)]  # Example input tensor\n\n\ndef get_init_inputs():\n    return []  # No initialization parameters required\n```")

You are a PyTorch and CUDA expert. Accelerate the given PyTorch Model by creating a high-performance CUDA C++ extension, targeting the best possible performance faster than baseline.

⚠️ STRICTLY FORBIDDEN
NO torch operators in C++: NEVER use torch::* or torch::nn::functional::* in binding.cpp or .cu files
NO torch operations in model_new.py: Only tensor creation and your custom ops allowed
NO third-party libraries: Except cuBLAS (GEMM only) and cuDNN (Conv only)

✅ ALLOWED ONLY
C++: Raw CUDA kernels (for custom ops), cuBLAS (for GEMM), cuDNN (MANDATORY for Conv/ConvTranspose)
Python: torch.tensor creation, custom extension ops, tensor properties (.shape, .device)
Memory: torch::empty_like for allocation only
Focus: Implement kernels in kernels/ directory only

WORKSPACE STRUCTURE
.
├── binding_registry.h    # Do NOT modify - registration system
├── binding.cpp           # Do NOT modify - main module binding
├── kernels/              # YOUR WORK: Implement all kernels here
├── model.py

In [25]:
cuda_agent_rl_data_v1 = pd.read_parquet("/nfs/FM/lihongbin/datasets/CUDA-Agent-Ops-6K/rl_data.parquet")

In [73]:
cuda_agent_rl_data_v2 = pd.read_parquet("/nfs/FM/lihongbin/datasets/CUDA_RL/cuda_agent_ops_6k.parquet")

In [23]:
cuda_agent_rl_data_v2.columns

Index(['data_source', 'prompt', 'ability', 'reward_model', 'extra_info'], dtype='object')

In [26]:
cuda_agent_rl_data_v1.columns

Index(['data_source', 'prompt', 'ability', 'reward_model', 'extra_info'], dtype='object')

In [74]:
i = random.randint(0, len(cuda_agent_rl_data_v2))
for k in ['reward_model', 'ability', 'data_source', 'prompt']:
    if k == "prompt":
        print("prompt len", len(cuda_agent_rl_data_v1.iloc[i][k]) == len(cuda_agent_rl_data_v2.iloc[i][k]))
    print(k, cuda_agent_rl_data_v1.iloc[i][k]==cuda_agent_rl_data_v2.iloc[i][k])

for k, v in cuda_agent_rl_data_v1.iloc[i]['extra_info'].items():
    print(k, cuda_agent_rl_data_v1.iloc[i]['extra_info'][k]==cuda_agent_rl_data_v2.iloc[i]['extra_info'][k])
 


reward_model True
ability True
data_source True
prompt len True
prompt [False]
entry_point True
level True
module_name True
ops True
original_prompt [False]
repo_name True
type True
uuid True


In [1]:
def check_dataset(data1, data2):
    assert len(data1) == len(data2)
    for i in range(len(data1)):
        try:
            
            check_pass = True
            check_pass = check_pass and len(data1.iloc[i]['prompt']) == len(data2.iloc[i]['prompt'])
            for k in ['reward_model', 'ability', 'data_source']:
                check_pass = check_pass and len(data1.iloc[i][k]) == len(data2.iloc[i][k])

            for k in data1.iloc[i]['extra_info']:
                if k == 'original_prompt':
                    continue
                check_pass = check_pass and data1.iloc[i]['extra_info'][k]==data2.iloc[i]['extra_info'][k]
        except Exception as e:
            print(i)
            raise e
        if not check_pass:
            print(i)
    print("check pass")

def check_label(dataset):
    for i in range(len(dataset)):
        check_pass = dataset.iloc[i]['reward_model']['ground_truth'] in dataset.iloc[i]['prompt'][0]['content']
        if not check_pass:
            print(i)
    print("label check pass")


In [ ]:
check_dataset(cuda_agent_rl_data_v1, cuda_agent_rl_data_v2)
check_label(cuda_agent_rl_data_v2)

In [76]:
print(cuda_agent_rl_data_v2.iloc[i]['prompt'][0]['content'])

You are a PyTorch and CUDA expert. Accelerate the given PyTorch Model by creating a high-performance CUDA C++ extension, targeting the best possible performance faster than baseline.

⚠️ STRICTLY FORBIDDEN
NO torch operators in C++: NEVER use torch::* or torch::nn::functional::* in binding.cpp or .cu files
NO torch operations in model_new.py: Only tensor creation and your custom ops allowed
NO third-party libraries: Except cuBLAS (GEMM only) and cuDNN (Conv only)

✅ ALLOWED ONLY
C++: Raw CUDA kernels (for custom ops), cuBLAS (for GEMM), cuDNN (MANDATORY for Conv/ConvTranspose)
Python: torch.tensor creation, custom extension ops, tensor properties (.shape, .device)
Memory: torch::empty_like for allocation only
Focus: Implement kernels in kernels/ directory only

WORKSPACE STRUCTURE
Write only the following output sections:
- CUDA_KERNELS: CUDA kernels and helper/device code.
- APPLY_BINDINGS: C++/pybind bindings and launcher wrappers.
- MODEL_NEW: Python ModelNew implementation using cu

In [ ]:
drkernel_rl_data_pv4 = pd.read_parquet("/nfs/FM/lihongbin/datasets/CUDA_RL/RL_Data/prompt_v4/drkernel_rl_thinking.parquet")
drkernel_rl_data_pmix = pd.read_parquet("/nfs/FM/lihongbin/datasets/CUDA_RL/RL_Data/prompt_mix/drkernel_rl_thinking.parquet")

In [12]:
check_dataset(drkernel_rl_data, drkernel_rl_data_pv4)

check pass


In [11]:
check_label(drkernel_rl_data_pv4)

label check pass


In [93]:
print(drkernel_rl_data_v2.iloc[i]['prompt'][0]['content'])

You are a PyTorch and CUDA expert. Accelerate the given PyTorch Model by creating a high-performance CUDA C++ extension, targeting the best possible performance faster than baseline.

⚠️ STRICTLY FORBIDDEN
NO torch operators in C++: NEVER use torch::* or torch::nn::functional::* in binding.cpp or .cu files
NO torch operations in model_new.py: Only tensor creation and your custom ops allowed
NO third-party libraries: Except cuBLAS (GEMM only) and cuDNN (Conv only)

✅ ALLOWED ONLY
C++: Raw CUDA kernels (for custom ops), cuBLAS (for GEMM), cuDNN (MANDATORY for Conv/ConvTranspose)
Python: torch.tensor creation, custom extension ops, tensor properties (.shape, .device)
Memory: torch::empty_like for allocation only
Focus: Implement kernels in kernels/ directory only

WORKSPACE STRUCTURE
Write only the following output sections:
- CUDA_KERNELS: CUDA kernels and helper/device code.
- APPLY_BINDINGS: C++/pybind bindings and launcher wrappers.
- MODEL_NEW: Python ModelNew implementation using cu

In [94]:
py_code = """import torch
import torch.nn as nn


class ModelNew(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding, dilation, bias):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.dilation = dilation
        self.bias = bias

        self.weight = nn.Parameter(torch.randn(out_channels, in_channels, kernel_size, kernel_size))
        self.bias = nn.Parameter(torch.zeros(out_channels)) if bias else None

    def forward(self, x):
        batch, channels, height, width = x.shape

        kernel_size = self.kernel_size
        pad = self.padding
        stride = self.stride
        dilation = self.dilation

        // Calculate output dimensions
        out_height = (height + 2 * pad - dilation * (kernel_size - 1) - 1) // stride + 1
        out_width = (width + 2 * pad - dilation * (kernel_size - 1) - 1) // stride + 1

        output = torch.empty((batch, self.out_channels, out_height, out_width),
                             dtype=x.dtype, device=x.device, requires_grad=x.requires_grad)

        // Execute convolution
        convolution_forward(
            x.data_ptr(), self.weight.data_ptr(), output.data_ptr(),
            batch, channels, height, width,
            self.out_channels, kernel_size, pad, stride, dilation,
            torch.cuda.current_stream()
        )

        if self.bias is not None:
            output = output + self.bias.view(1, -1, 1, 1)

        // Apply log2 transformation
        result = log2_forward(output)

        return result
"""

In [95]:
compile(py_code, filename="model_new.py", mode="exec")

SyntaxError: invalid syntax (model_new.py, line 27)

In [15]:
opus_sft.head()

,messages,think,response,prompt,source_messages,uuid,source_dataset
0,[{'content': 'I'm implementing an API gateway ...,The user is asking about implementing an API g...,# API Gateway Architecture for 20 Microservice...,I'm implementing an API gateway for 20 microse...,[{'content': 'I'm implementing an API gateway ...,0,TeichAI/Claude-Opus-4.6-Reasoning-887x
1,[{'content': 'How do we measure and manage tec...,The user is asking about measuring and managin...,\n\n# Measuring and Managing Technical Debt\n\...,How do we measure and manage technical debt in...,[{'content': 'How do we measure and manage tec...,1,TeichAI/Claude-Opus-4.6-Reasoning-887x
2,[{'content': 'Design a system for handling fil...,The user wants me to design a comprehensive sy...,# File Upload System at Scale — Complete Desig...,Design a system for handling file uploads at s...,[{'content': 'Design a system for handling fil...,2,TeichAI/Claude-Opus-4.6-Reasoning-887x
3,[{'content': 'Build a JWT auth system with log...,The user wants a JWT authentication system wit...,\n\n# JWT Authentication System (Node.js + Exp...,"Build a JWT auth system with login, signup, an...",[{'content': 'Build a JWT auth system with log...,3,TeichAI/Claude-Opus-4.6-Reasoning-887x
4,[{'content': 'I need a database design for a s...,The user wants a database design for a survey ...,\n\n# Survey System Database Design\n\n## Enti...,I need a database design for a survey system w...,[{'content': 'I need a database design for a s...,4,TeichAI/Claude-Opus-4.6-Reasoning-887x


In [52]:
cuda_sft = pd.read_parquet("/nfs/FM/lihongbin/datasets/CUDA_RL/SFT/parallel_drkernel_minimax_results_sft.parquet")

In [54]:
print_messages(cuda_sft.iloc[0]['messages'])

Role: user
You are a PyTorch and CUDA expert. Accelerate the given PyTorch Model by creating a high-performance CUDA C++ extension, targeting the best possible performance faster than baseline.

⚠️ STRICTLY FORBIDDEN
NO torch operators in C++: NEVER use torch::* or torch::nn::functional::* in binding.cpp or .cu files
NO torch operations in model_new.py: Only tensor creation and your custom ops allowed
NO third-party libraries: Except cuBLAS (GEMM only) and cuDNN (Conv only)

✅ ALLOWED ONLY
C++: Raw CUDA kernels (for custom ops), cuBLAS (for GEMM), cuDNN (MANDATORY for Conv/ConvTranspose)
Python: torch.tensor creation, custom extension ops, tensor properties (.shape, .device)
Memory: torch::empty_like for allocation only
Focus: Implement kernels in kernels/ directory only

WORKSPACE STRUCTURE
Role: assistant
<think>Let me analyze the task:

1. I need to create a CUDA extension for the given PyTorch model
2. The model is simple: `x = abs(x) - 1.0`
3. I need to implement custom CUDA kerne

In [55]:
cuda_sft_v3 = pd.read_parquet("/nfs/FM/lihongbin/datasets/CUDA_RL/SFT/prompt_v3/parallel_drkernel_minimax_results_sft.parquet")
cuda_sft_v4 = pd.read_parquet("/nfs/FM/lihongbin/datasets/CUDA_RL/SFT/prompt_v4/parallel_drkernel_minimax_results_sft.parquet")
cuda_sft_sample_v3 = pd.read_parquet("/nfs/FM/lihongbin/datasets/CUDA_RL/SFT/prompt_v3/parallel_drkernel_minimax_results_sft_sample2000_round1_2_ratio0p8_0p2.parquet")
opus_sft = pd.read_parquet("/nfs/FM/lihongbin/datasets/CUDA_RL/SFT/claude_opus_4_6_reasoning_887x_sft.parquet")

In [50]:
def print_messages(messages):
    for msg in messages[:3]:
        print("Role: " + msg['role'])
        print("\n".join(msg['content'].split("\n")[:20]))
        print("="*50)


In [42]:
len(cuda_sft_sample)

2000

In [ ]:
print_messages(cuda_sft_v4.iloc[0]['messages'])

Role: user
You are a PyTorch and CUDA expert. Accelerate the given PyTorch Model by creating a high-performance CUDA C++ extension, targeting the best possible performance faster than baseline.

KEY RULES
FORBIDDEN:
- Do not use PyTorch/ATen compute shortcuts anywhere. No `torch::*`, `torch::nn::functional::*`, ATen compute ops, or Python torch ops in `model_new.py`.
- Do not use third-party libraries except cuBLAS for GEMM and cuDNN for Conv/ConvTranspose.

ALLOWED:
- C++: raw CUDA custom kernels, cuBLAS for GEMM, cuDNN for Conv/ConvTranspose.
- Python: `torch.tensor` creation, custom `cuda_extension` ops, and tensor properties such as `.shape` and `.device`.
- Memory: `torch::empty_like` for output allocation only.
- Bindings: tensor validation, metadata/data pointers, and current CUDA stream access.

WORKFLOW:
- Implement core computation with real CUDA `__global__` kernels, then expose them through C-interface launchers and pybind wrappers.
- Output exactly three sections: CUDA_KER

In [43]:

cuda_sft.head()

,messages,uuid,entry_point,repo_name,module_name,final_speedup,num_rounds,original_python_code,best_round,timestamp,conversion_mode,enable_thinking
0,[{'content': 'You are a PyTorch and CUDA exper...,0,Model,,,1.780899,5,import torch\n\n\nclass Model(torch.nn.Module)...,5,1.776078e+09,full_conversation_enhanced,False
1,[{'content': 'You are a PyTorch and CUDA exper...,1,Model,,,1.703911,1,import torch\nimport torch.nn as nn\n\n\nclass...,1,1.776078e+09,full_conversation_enhanced,False
2,[{'content': 'You are a PyTorch and CUDA exper...,10,Model,,,1.637363,4,import torch\nimport torch.nn as nn\n\n\nclass...,4,1.776080e+09,full_conversation_enhanced,False
3,[{'content': 'You are a PyTorch and CUDA exper...,1000,Model,,,1.513966,1,import torch\n\n\nclass Model(torch.nn.Module)...,1,1.776126e+09,full_conversation_enhanced,False
4,[{'content': 'You are a PyTorch and CUDA exper...,1001,Model,,,2.308140,3,import torch\n\n\nclass Model(torch.nn.Module)...,3,1.776127e+09,full_conversation_enhanced,False
